In [4]:
import pandas as pd
data = pd.read_csv("files/selected_features_data.csv")
print(data.head())

        Ram  CPU_freq  PrimaryStorage    Weight   ScreenW   ScreenH  \
0  0.096774  0.518519        0.058824  0.169576  0.482619  0.597701   
1  0.096774  0.333333        0.058824  0.162095  0.029911  0.094828   
2  0.096774  0.592593        0.121569  0.291771  0.223929  0.224138   
3  0.225806  0.666667        0.247059  0.284289  0.611964  0.741379   
4  0.096774  0.814815        0.121569  0.169576  0.482619  0.597701   

   TypeName_Notebook  PrimaryStorageType_SSD    Inches  \
0                0.0                     1.0  0.385542   
1                0.0                     0.0  0.385542   
2                1.0                     1.0  0.662651   
3                0.0                     1.0  0.638554   
4                0.0                     1.0  0.385542   

   PrimaryStorageType_HDD  Screen_Standard  Screen_Full HD  Price_euros  
0                     0.0              1.0             0.0     0.196741  
1                     0.0              1.0             0.0     0.122353  
2 

# Splitting Data

In [5]:
from sklearn.model_selection import train_test_split

y = data['Price_euros']  
X = data.drop('Price_euros', axis=1) 

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (1020, 12)
X_test shape: (255, 12)
y_train shape: (1020,)
y_test shape: (255,)


In [6]:
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error


results = {
    "Model": [],
    "R² Score": [], # measures how well the model's predictions match the actual data
    "MAE": [], # average absolute difference between predicted and actual values
    "MSE": [] # average squared difference between predicted and actual values
}

def evaluate_model(model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    print(f"Model: {model.__class__.__name__}")
    print(f"R² Score: {r2:.4f}")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f} \n")
    results["Model"].append(model.__class__.__name__)
    results["MAE"].append(mae)
    results["MSE"].append(mse)
    return r2, mae, mse

# Linear Regression
lr_model = LinearRegression()
evaluate_model(lr_model, X_train, X_test, y_train, y_test)

# Random Forest
rf_model = RandomForestRegressor(random_state=42)
evaluate_model(rf_model, X_train, X_test, y_train, y_test)

# Gradient Boosting
gb_model = GradientBoostingRegressor(random_state=42)
evaluate_model(gb_model, X_train, X_test, y_train, y_test)

Model: LinearRegression
R² Score: 0.7126
MAE: 0.0475
MSE: 0.0041 

Model: RandomForestRegressor
R² Score: 0.8216
MAE: 0.0345
MSE: 0.0025 

Model: GradientBoostingRegressor
R² Score: 0.8124
MAE: 0.0372
MSE: 0.0027 



(0.8123965570390504, 0.037203568444025066, 0.002652422705013813)

In [7]:
from sklearn.model_selection import cross_val_predict


def cross_val(model,  X_train, y_train, cv=5):
    y_pred_cv = cross_val_predict(model, X_train, y_train, cv=cv)
    r2_cv = r2_score(y_train, y_pred_cv)  
    mae_cv = mean_absolute_error(y_train, y_pred_cv)
    mse_cv = mean_squared_error(y_train, y_pred_cv)
    print(f"Cross-validated results for {model.__class__.__name__}:")
    print(f"  R²: {r2_cv:.4f}")
    print(f"  MAE: {mae_cv:.4f}")
    print(f"  MSE: {mse_cv:.4f}")
    
    return r2_cv, mae_cv, mse_cv

cross_val(lr_model, X, y)
cross_val(rf_model, X, y)
cross_val(gb_model, X, y)

Cross-validated results for LinearRegression:
  R²: 0.6954
  MAE: 0.0471
  MSE: 0.0043
Cross-validated results for RandomForestRegressor:
  R²: 0.7654
  MAE: 0.0373
  MSE: 0.0033
Cross-validated results for GradientBoostingRegressor:
  R²: 0.7671
  MAE: 0.0391
  MSE: 0.0033


(0.7671151881617966, 0.03906852543054501, 0.003255015167062081)

# 4. Hyperparameter Tuning

Here we look for the best hyperparameters for each model

In [8]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

param_grid_lr = {
    'fit_intercept': [True, False]  # allows the regression model to shift the fitted line up or down to best fit the data
}

grid_search_lr = GridSearchCV(estimator=lr_model, param_grid=param_grid_lr, cv=5, scoring='r2', n_jobs=-1)
grid_search_lr.fit(X_train, y_train)
print("\nBest Random Forest Hyperparameters:", grid_search_lr.best_params_)

evaluate_model(grid_search_lr.best_estimator_, X_train, X_test, y_train, y_test)  


Best Random Forest Hyperparameters: {'fit_intercept': True}
Model: LinearRegression
R² Score: 0.7126
MAE: 0.0475
MSE: 0.0041 



(0.7126059240344422, 0.047530348042037034, 0.004063308009417418)

In [9]:
param_grid_rf = {
    'n_estimators': [100, 200, 500, 1000],           
    'max_depth': [20, 30, 40, None],          
    'min_samples_split': [2, 5, 10, 15],             
    'min_samples_leaf': [1, 2, 4, 6],         
}

grid_search_rf = GridSearchCV(estimator=rf_model, param_grid=param_grid_rf, cv=5, scoring='r2', n_jobs=-1)
grid_search_rf.fit(X_train, y_train)
print("\nBest Random Forest Hyperparameters:", grid_search_rf.best_params_)
evaluate_model(grid_search_rf.best_estimator_, X_train, X_test, y_train, y_test)  


Best Random Forest Hyperparameters: {'max_depth': 20, 'min_samples_leaf': 2, 'min_samples_split': 2, 'n_estimators': 1000}
Model: RandomForestRegressor
R² Score: 0.8084
MAE: 0.0351
MSE: 0.0027 



(0.8083791179700487, 0.03511139578092932, 0.0027092230836979494)

In [10]:
param_grid_gb = {
    'n_estimators': [100, 200, 500, 1000],  
    'learning_rate': [0.01, 0.05, 0.1, 0.2], 
    'max_depth': [3, 5, 8, 10],
    'subsample': [0.7, 0.8, 0.9, 1.0]
}

grid_search_gb = GridSearchCV(estimator=gb_model, param_grid=param_grid_gb, cv=5, scoring='r2', n_jobs=-1)
grid_search_gb.fit(X_train, y_train)
print("\nBest Gradient Boosting Hyperparameters:", grid_search_gb.best_params_)
best_gb_model = grid_search_gb.best_estimator_

evaluate_model(grid_search_gb.best_estimator_, X_train, X_test, y_train, y_test) 


Best Gradient Boosting Hyperparameters: {'learning_rate': 0.01, 'max_depth': 5, 'n_estimators': 500, 'subsample': 0.7}
Model: GradientBoostingRegressor
R² Score: 0.8384
MAE: 0.0344
MSE: 0.0023 



(0.8384458974921221, 0.034387497621761166, 0.0022841252954468464)

## Stacking Regressor

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor

rf_best = RandomForestRegressor(**grid_search_rf.best_params_, random_state=42)
gb_best = GradientBoostingRegressor(**grid_search_gb.best_params_, random_state=42)


final_estimators = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

for name, estimator in final_estimators.items():
    # Set up the stacking model with the current final estimator
    stacking_model = StackingRegressor(
        estimators=[('rf', rf_best), ('gb', gb_best)],
        final_estimator=estimator
    )
    
    print(f"\n{name} - Stacking Model:")
    model_results = evaluate_model(stacking_model, X_train, X_test, y_train, y_test)


Linear Regression - Stacking Model:
Model: StackingRegressor
R² Score: 0.8307
MAE: 0.0340
MSE: 0.0024 


Random Forest Regressor - Stacking Model:
Model: StackingRegressor
R² Score: 0.7212
MAE: 0.0414
MSE: 0.0039 


Gradient Boosting Regressor - Stacking Model:
Model: StackingRegressor
R² Score: 0.7493
MAE: 0.0379
MSE: 0.0035 



In [12]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import StackingRegressor

gb_best = GradientBoostingRegressor(**grid_search_gb.best_params_, random_state=42)
lr_best = LinearRegression(**grid_search_lr.best_params_)


final_estimators = {
    "Linear Regression": LinearRegression(),
    "Random Forest Regressor": RandomForestRegressor(random_state=42),
    "Gradient Boosting Regressor": GradientBoostingRegressor(random_state=42)
}

for name, estimator in final_estimators.items():
    # Set up the stacking model with the current final estimator
    stacking_model = StackingRegressor(
        estimators=[('lr', lr_best), ('gb', gb_best)],
        final_estimator=estimator
    )
    
    print(f"\n{name} - Stacking Model:")
    model_results = evaluate_model(stacking_model, X_train, X_test, y_train, y_test)


Linear Regression - Stacking Model:
Model: StackingRegressor
R² Score: 0.8348
MAE: 0.0351
MSE: 0.0023 


Random Forest Regressor - Stacking Model:
Model: StackingRegressor
R² Score: 0.7789
MAE: 0.0375
MSE: 0.0031 


Gradient Boosting Regressor - Stacking Model:
Model: StackingRegressor
R² Score: 0.8114
MAE: 0.0350
MSE: 0.0027 



Gradient Boosting is the best . An R² score of 0.8393 from cross-validation indicates that the model explains approximately 83.93% of the variance in the target variable.